# 03 LoRA Explained

## Overview

This notebook explains why LoRA matters and what its core settings do.
You will see a tiny matrix example, then attach adapters to a real language model and inspect the parameter counts.

Estimated time: 25 to 35 minutes.

## Prerequisites

Complete notebooks 00 through 02 first.
You should have a basic feel for the dataset format and model choice.

In [ ]:
%pip install -q peft==0.10.0 transformers==4.40.0 torch==2.2.2

## The Core Problem

Full fine tuning updates every trainable weight in the model.
That becomes expensive fast.

LoRA solves this by keeping the big base weights frozen and learning a much smaller update through adapter matrices.

In [ ]:
import numpy as np

base_weight = np.array([[1.0, 0.0], [0.5, 1.5]])
lora_a = np.array([[0.2], [0.1]])
lora_b = np.array([[0.3, -0.4]])
rank_update = lora_a @ lora_b
adapted_weight = base_weight + rank_update

print("Base weight:\n", base_weight)
print("\nLow rank update:\n", rank_update)
print("\nAdapted weight:\n", adapted_weight)

## Plain English Meaning of the Main LoRA Settings

* `r`: the rank of the adapter. Higher rank means more adapter capacity.
* `lora_alpha`: a scaling factor for the adapter update.
* `lora_dropout`: dropout applied inside the adapter path during training.
* `target_modules`: which layers get the adapter weights.

## CI Friendly Model Choice

This notebook uses a tiny Llama shaped test model so the adapter wiring stays fast on CPU.
The real training notebooks later in the repo switch to TinyLlama 1.1B.

In [ ]:
from pathlib import Path
import sys

sys.path.append(str(Path("..").resolve()))

from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM
from utils.helpers import print_trainable_parameters

model_name = "hf-internal-testing/tiny-random-LlamaForCausalLM"
base_model = AutoModelForCausalLM.from_pretrained(model_name)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"],
)

peft_model = get_peft_model(base_model, lora_config)
print_trainable_parameters(peft_model)

In [ ]:
sample_names = [name for name, parameter in peft_model.named_parameters() if parameter.requires_grad][:10]
sample_names

## Key Takeaways

* LoRA trains a small update instead of the full model.
* The low rank trick cuts memory use and training cost.
* `r`, `lora_alpha`, `lora_dropout`, and `target_modules` are the main knobs to understand first.
* PEFT makes LoRA setup much simpler than hand wiring adapters.

## Next Steps

Continue to [04 Your First Finetune](../04-your-first-finetune/04-your-first-finetune.ipynb).